# Semana 3 — Diagnóstico de overfitting y underfitting

**Estado: preparación, NO listo para entregar.** Falta seleccionar una tarea supervisada compatible con la consigna y disponer de datos etiquetados. No hay entrenamiento ni métricas ejecutadas.

Equipo: Schubert Lombeida Manjarrez y Niko Dimitri Jiménez Bruno.

Este es el archivo único de trabajo para semana 3. No confundir con el notebook de recuperación E5. Actualizar este mismo archivo cuando estén resueltas las decisiones.

## 1. Objetivo y definición de tarea
Pendiente: objetivo supervisado, unidad de observación, significado de etiqueta, procedencia, autorización y criterio de éxito. No usar relevancia generada por el buscador como verdad de referencia. No usar el conjunto de prueba para elegir estrategias.


## 2. Carga y exploración del dataset
El siguiente contrato inicial sirve para clasificación de textos o pares consulta-evidencia. Si el docente permite otra tarea, adaptar esta sección antes de entrenar.

CSV requerido: `sample_id`, `group_id`, `text`, `label`, `split`. `group_id` debe identificar el expediente o consulta que no puede cruzar particiones; `split` admite development, validation, test. Las etiquetas deben proceder de una fuente documentada o revisión humana.


In [ ]:
from pathlib import Path
import csv, hashlib
from collections import Counter, defaultdict
DATASET = Path('datos_semana3.csv')
LABEL_PROVENANCE = ''  # Completar procedencia y revisión de etiquetas.

def validate_dataset(path, provenance):
    if not provenance.strip():
        raise ValueError('Documentar procedencia de etiquetas antes de entrenar.')
    with path.open(encoding='utf-8-sig', newline='') as source:
        reader = csv.DictReader(source)
        required = {'sample_id', 'group_id', 'text', 'label', 'split'}
        if not required <= set(reader.fieldnames or []):
            raise ValueError('Faltan columnas requeridas.')
        rows = list(reader)
    if not rows:
        raise ValueError('Dataset vacío.')
    ids, groups, texts = set(), defaultdict(set), defaultdict(set)
    for row in rows:
        if any(not row[k].strip() for k in required):
            raise ValueError('Hay campos requeridos vacíos.')
        if row['sample_id'] in ids:
            raise ValueError('sample_id duplicado.')
        ids.add(row['sample_id'])
        if row['split'] not in {'development','validation','test'}:
            raise ValueError('Partición no reconocida.')
        groups[row['group_id']].add(row['split'])
        normalized = ' '.join(row['text'].lower().split())
        texts[normalized].add(row['split'])
    if any(len(v) > 1 for v in groups.values()):
        raise ValueError('Un grupo aparece en varias particiones.')
    if any(len(v) > 1 for v in texts.values()):
        raise ValueError('Texto duplicado entre particiones.')
    train = [r for r in rows if r['split'] == 'development']
    val = [r for r in rows if r['split'] == 'validation']
    if not train or not val or len({r['label'] for r in train}) < 2:
        raise ValueError('Se requiere desarrollo con dos clases y validación no vacía.')
    if not {r['label'] for r in val} <= {r['label'] for r in train}:
        raise ValueError('Validación contiene clases ausentes en desarrollo.')
    return train, val

if DATASET.exists() and LABEL_PROVENANCE.strip():
    train_rows, val_rows = validate_dataset(DATASET, LABEL_PROVENANCE)
    print('Desarrollo:', len(train_rows), Counter(r['label'] for r in train_rows))
    print('Validación:', len(val_rows), Counter(r['label'] for r in val_rows))
    print('SHA256:', hashlib.sha256(DATASET.read_bytes()).hexdigest())
else:
    print('PENDIENTE: cargar dataset y documentar procedencia de etiquetas. No se entrenó.')


## 3. Sistema de tracking
Registrar semilla, configuración, versiones, partición, duración y métricas por época. Para clasificación: pérdida y accuracy/score en desarrollo y validación; macro-F1 si hay desbalance. Ajustar vectorizador y transformaciones exclusivamente con desarrollo.


In [ ]:
import json, math

def save_history(history, path):
    required = {'epoch','train_loss','val_loss','train_score','val_score','seconds'}
    if not history:
        raise ValueError('No hay entrenamiento registrado.')
    previous = 0
    for row in history:
        if not required <= row.keys():
            raise ValueError('Tracking incompleto.')
        if row['epoch'] <= previous:
            raise ValueError('Las épocas deben ser crecientes.')
        previous = row['epoch']
        if any(not math.isfinite(float(row[k])) for k in required):
            raise ValueError('Métrica no finita.')
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(history, indent=2), encoding='utf-8')


## 4. Entrenamiento base con logging
**Pendiente de tarea y dataset.** Seleccionar un modelo entrenable apropiado. Registrar métricas sin forzar sobreajuste ni alterar las etiquetas para producir curvas. Conservar pesos/configuración del baseline.

## 5. Curvas requeridas
La función siguiente consume registros reales: loss y score train/validation. No crea valores artificiales. Exporta PNG a 300 DPI. El punto de menor pérdida de validación se anota; eso por sí solo no demuestra sobreajuste.


In [ ]:
def plot_history(history, output_dir):
    import matplotlib.pyplot as plt
    if not history:
        raise ValueError('Primero ejecutar entrenamiento real.')
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    epochs = [r['epoch'] for r in history]
    for metric, ylabel in [('loss','Pérdida'), ('score','Score (definir métrica en el informe)')]:
        fig, ax = plt.subplots(figsize=(8,4.5))
        ax.plot(epochs, [r['train_'+metric] for r in history], label='Entrenamiento', color='#145c78')
        ax.plot(epochs, [r['val_'+metric] for r in history], label='Validación', color='#b45309')
        best = min(range(len(history)), key=lambda i: history[i]['val_loss'])
        ax.axvline(epochs[best], linestyle='--', color='#555', label='Menor pérdida de validación')
        ax.set(xlabel='Época', ylabel=ylabel, title='Entrenamiento y validación: '+ylabel)
        ax.grid(alpha=.25); ax.legend(); fig.tight_layout()
        fig.savefig(output_dir / (metric+'.png'), dpi=300)
        plt.close(fig)


## 6. Análisis y diagnóstico
Pendiente: describir tendencias, brecha train/validation y evolución de la pérdida. Comparar score con un predictor trivial. Considerar tamaño muestral, desbalance y variabilidad; una brecha pequeña no demuestra un buen modelo.

## 7. Dos estrategias basadas en el diagnóstico
Elegir después del baseline. Si hay sobreajuste, considerar regularización y reducción de complejidad o early stopping; si hay subajuste, revisar representación y capacidad. Mantener partición y métrica comparables. No denominar «mejora» a una intervención que empeore los resultados.

## 8. Comparación antes/después
Pendiente: tabla de baseline y dos estrategias, curvas, configuración, tiempo y brechas. Una tabla vacía no sustituye los experimentos. Documentar resultados desfavorables.

## 9. Conclusiones y reflexiones
Pendiente de resultados reales. Explicar limitaciones y pasos siguientes. No extrapolar resultados de una tarea auxiliar a Recall@5 del RAG.

## 10. Entrega
- Este notebook ejecutado de inicio a fin, con salidas revisadas.
- `diagnostic_report.pdf`: resumen ejecutivo de una página más metodología, diagnóstico, curvas, estrategias/resultados, conclusiones y referencias. La consigna no limita todo el informe a una página.
- Figuras de al menos 300 DPI.
- Código modular y registros de ejecución.

Ponderación: tracking 25 %, curvas 30 %, diagnóstico 25 %, estrategias 20 %.
